## Practical 4: Estimation


### Exercise 1: Wheeled motion model calibration

We have a robot model

$$\begin{bmatrix}
\dot{x} \\
\dot{y} \\
\dot{\theta}
\end{bmatrix} = \begin{bmatrix}\frac{r}{2}(\omega_r + \omega_l) \cos(\theta) \\ \frac{r}{2}(\omega_r + \omega_l) \sin(\theta) \\ \frac{r}{L}(\omega_r - \omega_l)\end{bmatrix}$$

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets
from scipy.stats import norm
from matplotlib.lines import Line2D
import time

import sys
import os
sys.path.insert(0, os.path.abspath('Support'))

from Distribution import Distribution
from Helper import controls_from_lissajous, lissajous_reference, configure_live_plot, update_pose_artists, generate_measurements

In [ ]:
def model(x,u,theta):
    
    x_pos,y_pos,th = x
    speed, turn_rate = u
    r,L = theta
    
    dt = 0.1
    
    x_pos = x_pos + r*speed*np.cos(th)*dt
    y_pos = y_pos + r*speed*np.sin(th)*dt
    th = (th + r/L*turn_rate*dt) % (2*np.pi)
    
    return np.array([x_pos,y_pos,th])
    

In [ ]:
# Generate some "real" robot data - we will add some random noise to simulate uncertainty
r = 0.1
L = 0.5
T = 1000 #Run for T steps

data = []
x = np.zeros((3,))
# u = np.array([0.5,1])
for j in range(T):
    
    u = np.array([0.5,0])
    x = model(x,u,[r,L]) + np.random.normal(0,0.001,(3,))
    
    data.append(np.hstack((x,u)))
data = np.array(data)    

plt.figure(figsize=(8,6))
plt.plot(data[:,0],data[:,1],label='Real robot path')
plt.xlim(-2,2)
plt.ylim(-2,2)
plt.legend()
plt.show()

In [ ]:
# Lets use an exhaustive search to find the best parameters for our model. 
# We will use a grid search over a range of values for r and L, and compute 
# the mean squared error between the predicted and actual robot paths.

def wrap_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

histogram = np.zeros((20,20))
for i,r in enumerate(np.linspace(0.05,0.15,20)):
    for j,L in enumerate(np.linspace(0.4,0.6,20)):
        x = np.zeros((3,))
        pred = []
        for k in range(T-1):
            x = model(data[k, :3],data[k + 1, 3:5],[r,L])
            pred.append(x)
        pred = np.array(pred)
        position_mse = np.mean((pred[:,:2] - data[1:,:2])**2)
        angle_mse = np.mean(wrap_angle(pred[:,2] - data[1:,2])**2)
        histogram[i, j] = position_mse + angle_mse

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(histogram, extent=[0.4,0.6,0.05,0.15], origin='lower', aspect='auto')
plt.xlabel('L')
plt.ylabel('r')
plt.colorbar(label='MSE')
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(np.exp(-histogram), extent=[0.4,0.6,0.05,0.15], origin='lower', aspect='auto')
plt.xlabel('L')
plt.ylabel('r')
plt.colorbar(label='Likelhood')
plt.show()

 
Lets look at the likelihood for just the radius parameter r, by **marginalising** over L. 

We can do this by summing the likelihoods over all values of L for each value of r.

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
plt.plot(np.linspace(0.05,0.15,20), np.sum(np.exp(-histogram), axis=1))
plt.xlabel('r')
plt.ylabel('Marginalised Likelihood (unormalised)')
plt.subplot(1,2,2)
plt.plot(np.linspace(0.4,0.6,20), np.sum(np.exp(-histogram), axis=0))
plt.xlabel('L')
plt.ylabel('Marginalised Likelihood (unormalised)')
plt.show()

**Question:**

Which set of parameters maximise the likelihood?

Why are many parameters of L equally likely?


**Activity:** Change the trajectory to **excite** the system and allow for unique identification of the parameters.

In [ ]:
best_r = np.mean(np.linspace(0.05,0.15,20)[np.unravel_index(np.argmax(np.exp(-histogram)), histogram.shape)[0]])
best_L = np.mean(np.linspace(0.4,0.6,20)[np.unravel_index(np.argmax(np.exp(-histogram)), histogram.shape)[1]])

print(f'Best r: {best_r}, Best L: {best_L}')

## Exercise 2: Model Uncertainty: How?

### 1. Propagation of Uncertainty

Given the initial known position of our 1D robot ($x_t$), after each step, our uncertainty about the robot's new state is defined as follows:

$$
\begin{split}
& P(x_{t+1}=x_{t}+1) = 0.5, \\
& P(x_{t+1}=x_{t}+1.1) = 0.25, \\
& P(x_{t+1}=x_{t}+0.9) = 0.25. 
\end{split}
$$


Below we show how our uncertainty about the robot's true position changes with every new step.

**Interaction**
- Observe how our uncertainty about the robot's position changes when the robot moves forward 5 steps

In [ ]:
# We start with a known position: probability 1.0 at position 10
belief_position = Distribution.unit_pulse(10)
x, y = belief_position.plotlists(0, 500)

fig, ax = plt.subplots()
old_line, = ax.step(x, y, c='b', label='Current probability')
line, = ax.step(x, y, c='g', label='Probability after 5 steps')
ax.legend()
ax.set_xlabel("Robot's position along the 1D line")
ax.set_ylabel("Probability of robot being at position x")

def convolve_distribution(b=None):
    global belief_position
    
    # For each new control:
    # - there is 50% chance the robot moves 1m
    # - there is 25% chance the robot moves 1.1m
    # - there is 25% change the robot moves 0.9m
    
    # Plot prior belief
    old_x, old_y = belief_position.plotlists(0, 500)
    old_line.set_data(old_x, old_y)
    
    # We enconde this belief into a new distribution
    for i in range(5):
        move_distribution = Distribution.triangle(1,2)
        # We modify our belief about the robots position with a convolution
        belief_position = belief_position.convolve(move_distribution)
        new_x, new_y = belief_position.plotlists(0, 500)
        line.set_data(new_x, new_y)
        
def reset(b=None):
    global belief_position
    belief_position = Distribution.unit_pulse(10)
    x, y = belief_position.plotlists(0, 500)
    old_line.set_data(x,y)
    line.set_data(x, y)
    

btn_move = widgets.Button(description='Move 5 steps', 
                          layout=widgets.Layout(flex='1 1 0%', width='auto'),
                          button_style='success')
btn_reset = widgets.Button(description='Reset', 
                          layout=widgets.Layout(flex='1 1 0%', width='auto'),
                          button_style='success')

btn_move.on_click(convolve_distribution)
btn_reset.on_click(reset)

display(widgets.HBox([btn_move, btn_reset]))

### 2. Reducing Uncertainty with Measurements - 1D Kalman Filter

Lets see how we can combine our model predictions and the noisy measurements from the robot's sensor in order to reduce our uncertainty about the robot's position. 

The robot model is defined as $x_{k+1} = ax_k + bu_k + \epsilon_Q, \epsilon_Q \sim \mathcal{N}(0, \sigma^2_Q)$. Similarly, we define the measurement model as $z_k = cx_k + \epsilon_R, \epsilon_R \sim \mathcal{N}(0, \sigma^2_R)$. 

### Step 1: 
We generate the true states and the noisy measurements associated to those states.

In [ ]:
# 1D model parameters
u = 100 # We will drive our robot with a constant control input
a = 1
b = 1

# Number of measurements we want to generate
steps = 10

# This is our measurement constant
C = 1

#This is our true noise
true_move_noise = 2
true_mes_noise = 10

true_state, measurements = generate_measurements(steps=steps, a=a, b=b, u=u, c=C, true_move_noise=true_move_noise,
                                                true_mes_noise=true_mes_noise)

In [ ]:
#----------------------------------PARAMETERS--------------------------------------
# This is our mean at t, we initialize at zero
mu_k = 0
# This is our covariance at t, we initialize very small implying we know x0 very well
sigma_k = 0.4
# This is the uncertainty in our move function 
sigma_Q = 10
# This is what we think our measurement noise is
sigma_R = 10
#-----------------------------------------------------------------------------------

# We will save our predicted state here
pred_state = np.zeros((steps,1)) 
# We will save our estimated state here
est_state = np.zeros((steps,1))

# Plotting code
x = np.linspace(0,1000,2000)
y_pred = norm.pdf(x, loc=mu_k, scale=sigma_k)
y_mes = norm.pdf(x, loc=true_state[0], scale=sigma_R)
y_est = norm.pdf(x, loc=mu_k, scale=sigma_k)

# If this cell is re-run, close previous figures to avoid duplicate outputs in VS Code.
plt.close('all')
fig, ax = plt.subplots()
pred_curve, = ax.plot(x, y_pred, c='g')
mes_curve, = ax.plot(x, y_mes, c='r')
est_curve, = ax.plot(x, y_est, c='orange')
true_scatter = ax.scatter([], [], color='b')

ax.set_xlabel("Robot's position along the 1D line")
ax.set_ylabel("Uncertainty on robot's position")

legend_elements = [Line2D([0], [0], color='g', lw=2, label='Model Uncertainty'),
                   Line2D([0], [0], color='r', lw=2, label='Measurement Uncertainty'),
                   Line2D([0], [0], color='orange', lw=2, label='KF Uncertainty'),
                   Line2D([0], [0], marker='o', color='w', label='True Position',
                          markerfacecolor='b', markersize=8)]

ax.legend(handles=legend_elements, loc='upper right')

# Force updates into one notebook output area instead of creating a new render each draw.
display_handle = display(fig, display_id=True)

true_x = []
true_y = []

c = 1
for i in range(steps-1):
    
    # Prediction step
    mu_bar = a*mu_k + b*u
    sigma_bar = np.sqrt(a*a*sigma_k*sigma_k + sigma_Q*sigma_Q)  

    # Update step
    z = measurements[i+1]
    #-------------------- TODO: Complete this step----------------
    # 1. Compute the Kalman Gain
    K = c*sigma_bar*sigma_bar/(c*c*sigma_bar*sigma_bar + sigma_R*sigma_R)
    # 2. Apply correction to mu_k
    mu_k = mu_bar + K*(z - c*mu_bar)
    # 3. Apply correction to sigma_k
    sigma_k = np.sqrt((1 - K*c)*sigma_bar*sigma_bar)
    #---------------------------------------------------------------#

    # Update plotted distributions instead of plotting new artists each iteration
    y_pred = norm.pdf(np.array(x), loc=mu_bar, scale=sigma_bar)
    y_mes = norm.pdf(np.array(x), loc=measurements[i+1], scale=sigma_R)
    y_est = norm.pdf(np.array(x), loc=mu_k, scale=sigma_k)

    pred_curve.set_data(x, y_pred)
    mes_curve.set_data(x, y_mes)
    est_curve.set_data(x, y_est)

    # Save our estimates
    pred_state[i+1] = mu_bar
    est_state[i+1] = mu_k

    true_x.append(true_state[i+1])
    true_y.append(0)
    true_scatter.set_offsets(np.c_[true_x, true_y])

    display_handle.update(fig)
    time.sleep(0.5)


## Exercise 3: EKF sensor fusion

We use the state

$$
\mathbf{x}_k = [x_k,\;y_k,\;\theta_k,\;v_k]^T
$$

and the action

$$
\mathbf{u}_k = [v_k^{cmd},\;\theta_k^{cmd}]^T.
$$

The second action is a **desired heading angle**, not a turn rate. To model the fact that the robot does not instantaneously reach its commanded speed and heading, use a simple first-order response model:

$$
\begin{aligned}
x_{k+1} &= x_k + v_k\cos(\theta_k)\,dt,\\
y_{k+1} &= y_k + v_k\sin(\theta_k)\,dt,\\
\theta_{k+1} &= \theta_k + \alpha_\theta\,\mathrm{wrap}(\theta_k^{cmd}-\theta_k),\\
v_{k+1} &= v_k + \alpha_v(v_k^{cmd}-v_k).
\end{aligned}
$$

The parameters $\alpha_v$ and $\alpha_\theta$ describe how quickly the robot responds to commands. These parameters, together with $Q$ and $R$, should be calibrated against the real robot.

In [ ]:
# Important to always take care of angle wrapping when computing errors between angles.
def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


# Four-state motion model: state = [x, y, heading, speed].
def motion_model(state, control, dt, alpha_speed, alpha_heading):
    """Four-state motion model: state = [x, y, heading, speed]."""
    x, y, heading, speed = state
    speed_cmd, heading_cmd = control

    next_state = np.array([
        x + speed * np.cos(heading) * dt,
        y + speed * np.sin(heading) * dt,
        wrap_angle(
            heading
            + alpha_heading * wrap_angle(heading_cmd - heading)
        ),
        speed + alpha_speed * (speed_cmd - speed),
    ])
    return next_state

# df/dx for the four-state motion model.
def motion_jacobian_state(state, dt, alpha_speed, alpha_heading):
    """Jacobian df/dx for the four-state motion model."""
    _, _, heading, speed = state

    return np.array([
        [
            1.0,
            0.0,
            -speed * np.sin(heading) * dt,
            np.cos(heading) * dt,
        ],
        [
            0.0,
            1.0,
            speed * np.cos(heading) * dt,
            np.sin(heading) * dt,
        ],
        [0.0, 0.0, 1.0 - alpha_heading, 0.0],
        [0.0, 0.0, 0.0, 1.0 - alpha_speed],
    ])

# EKF predict and update functions.
def ekf_predict(mean, covariance, control, dt, alpha_speed,
                alpha_heading, process_noise):
    F = motion_jacobian_state(
        mean, dt, alpha_speed, alpha_heading
    )
    predicted_mean = motion_model(
        mean, control, dt, alpha_speed, alpha_heading
    )
    predicted_covariance = F @ covariance @ F.T + process_noise
    return predicted_mean, predicted_covariance


def ekf_update(predicted_mean, predicted_covariance, measurement,
               measurement_noise, measurement_model,
               measurement_jacobian, angle_measurement_indices=()):
    """Generic EKF update used by both demonstrations below."""
    H = measurement_jacobian(predicted_mean)
    innovation = measurement - measurement_model(predicted_mean)

    for index in angle_measurement_indices:
        innovation[index] = wrap_angle(innovation[index])

    innovation_covariance = (
        H @ predicted_covariance @ H.T + measurement_noise
    )
    kalman_gain = (
        predicted_covariance
        @ H.T
        @ np.linalg.inv(innovation_covariance)
    )

    updated_mean = predicted_mean + kalman_gain @ innovation
    updated_mean[2] = wrap_angle(updated_mean[2])

    # Joseph form helps preserve symmetry and positive semidefiniteness.
    identity = np.eye(len(predicted_mean))
    correction = identity - kalman_gain @ H
    updated_covariance = (
        correction @ predicted_covariance @ correction.T
        + kalman_gain @ measurement_noise @ kalman_gain.T
    )

    return updated_mean, updated_covariance


### Full-state observations

First assume that the sensor directly measures the complete state:

$$
\mathbf{z}_k = [x_k^{meas},\;y_k^{meas},\;\theta_k^{meas},\;v_k^{meas}]^T.
$$

Then

$$
h(\mathbf{x}_k)=\mathbf{x}_k, \qquad H_k=I_4.
$$


In [ ]:
def full_state_measurement_model(state):
    return state.copy()


def full_state_measurement_jacobian(state):
    return np.eye(4)


In [ ]:
rng = np.random.default_rng(seed=7)

dt = 0.05
total_time = 30.0
time = np.arange(0.0, total_time, dt)
number_of_steps = len(time)

commanded_speed, commanded_heading = controls_from_lissajous(time)
reference_x, reference_y, _, _ = lissajous_reference(time)

# True robot response parameters used by the simulator.
true_alpha_speed = 0.22
true_alpha_heading = 0.30

# Parameters assumed by the EKF.
model_alpha_speed = 0.18
model_alpha_heading = 0.25


In [ ]:

# Disturbances applied to the simulated true robot state.
true_process_std = np.array([
    0.01,
    0.01,
    np.deg2rad(1.0),
    0.05,
])

# Process covariance assumed by the EKF.
process_noise = np.diag(np.array([
    0.012,
    0.012,
    np.deg2rad(1.5),
    0.060,
]) ** 2)

# Measurement noise covariance
full_measurement_std = np.array([
    0.5,
    0.5,
    np.deg2rad(2.0),
    0.5,
])
full_measurement_noise = np.diag(full_measurement_std ** 2)

#Initial true state and initial EKF estimate.
true_state = np.array([
    reference_x[0],
    reference_y[0],
    commanded_heading[0],
    commanded_speed[0],
])

estimated_mean = true_state 
estimated_mean[2] = wrap_angle(estimated_mean[2])

# Initial covariance assumed by the EKF - very confident about the initial state. 
estimated_covariance = np.diag(np.array([
    0.05,
    0.05,
    np.deg2rad(0.05),
    0.05,
]) ** 2)


In [ ]:
true_history = np.zeros((number_of_steps, 4))
measurement_history = np.zeros((number_of_steps, 4))
estimate_history = np.zeros((number_of_steps, 4))
covariance_history = np.zeros((number_of_steps, 4, 4))

# Some plotting.
(
    fig,
    axis,
    display_handle,
    measurement_scatter,
    true_line,
    estimate_line,
    estimate_point,
    heading_line,
    pose_artists,
) = configure_live_plot(
    reference_x,
    reference_y,
    "EKF with full-state measurements\nEllipses show 2-sigma XY covariance",
    show_measurements=True,
)

update_every = 3

# Let's start driving.
for k in range(number_of_steps):
    control = np.array([
        commanded_speed[k],
        commanded_heading[k],
    ])

    # Simulated true robot dynamics.
    true_state = motion_model(
        true_state,
        control,
        dt,
        true_alpha_speed,
        true_alpha_heading,
    )
    true_state += rng.normal(0.0, true_process_std)
    true_state[2] = wrap_angle(true_state[2])

    # Full-state sensor observation.
    measurement = true_state + rng.normal(
        0.0, full_measurement_std
    )
    measurement[2] = wrap_angle(measurement[2])

    ######################### EKF Prediction
    estimated_mean, estimated_covariance = ekf_predict(
        estimated_mean,
        estimated_covariance,
        control,
        dt,
        model_alpha_speed,
        model_alpha_heading,
        process_noise,
    )

    ######################### EKF Update
    estimated_mean, estimated_covariance = ekf_update(
        estimated_mean,
        estimated_covariance,
        measurement,
        full_measurement_noise,
        full_state_measurement_model,
        full_state_measurement_jacobian,
        angle_measurement_indices=(2,),
    )

    # Plotting and history storage.
    true_history[k] = true_state
    measurement_history[k] = measurement
    estimate_history[k] = estimated_mean
    covariance_history[k] = estimated_covariance

    if k % update_every == 0 or k == number_of_steps - 1:
        true_line.set_data(
            true_history[:k + 1, 0], true_history[:k + 1, 1]
        )
        estimate_line.set_data(
            estimate_history[:k + 1, 0], estimate_history[:k + 1, 1]
        )
        estimate_point.set_data(
            [estimated_mean[0]], [estimated_mean[1]]
        )
        measurement_scatter.set_offsets(
            measurement_history[:k + 1:8, :2]
        )

        update_pose_artists(
            axis,
            heading_line,
            pose_artists,
            estimated_mean,
            estimated_covariance,
        )
        display_handle.update(fig)


## Exercise

1. Increase the measurement uncertainty in the x and y measurements and rerun the code. What happens?
2. Now keep this, and increase the process noise uncertainty in the y prediction from the model. What happens?
3. Now remove the update step from the EKF.


A good filter has good mean prediction (estimate error) and also good coverage, the uncertainty covers the range of possible states.




## Velocity-only observations

Now remove the direct position and heading observations. The sensor measures only robot speed:

$$
\mathbf{z}_k=[v_k^{meas}].
$$

The measurement model is

$$
h(\mathbf{x}_k)=v_k,
\qquad
H_k=\begin{bmatrix}0&0&0&1\end{bmatrix}.
$$

The speed measurement corrects the velocity state, which improves the integrated position estimate. However, there is no external reference for global position or heading, so the trajectory can still drift and the position covariance should grow.

Exercise: Update the Jacobian in the measurement model to only select velocity from the state.

Then run the EKF below and see what happens.

In [ ]:
def velocity_measurement_model(state):
    return np.array([state[3]])


def velocity_measurement_jacobian(state):
    return #Todo - change this


In [ ]:
rng = np.random.default_rng(seed=7)

#Set up trajectory and time parameters.
dt = 0.05
total_time = 30.0
time = np.arange(0.0, total_time, dt)
number_of_steps = len(time)

commanded_speed, commanded_heading = controls_from_lissajous(time)
reference_x, reference_y, _, _ = lissajous_reference(time)

# True robot response parameters used by the simulator.
true_alpha_speed = 0.22
true_alpha_heading = 0.30
model_alpha_speed = 0.18
model_alpha_heading = 0.25

# True noise applied to the simulated true robot state.
true_process_std = np.array([
    0.005,
    0.005,
    np.deg2rad(2.0),
    0.05,
])

# Process covariance assumed by the EKF.
process_noise = np.diag(np.array([
    0.012,
    0.012,
    np.deg2rad(1.5),
    0.060,
]) ** 2)

# Measurement noise covariance for the velocity sensor.
velocity_measurement_std = np.array([0.2])
velocity_measurement_noise = np.diag(
    velocity_measurement_std ** 2
)

# Initial true state and initial EKF estimate.
true_state = np.array([
    reference_x[0],
    reference_y[0],
    commanded_heading[0],
    commanded_speed[0],
])

estimated_mean = true_state
estimated_mean[2] = wrap_angle(estimated_mean[2])

# Initial covariance assumed by the EKF - very confident about the initial state.
estimated_covariance = np.diag(np.array([
    0.01,
    0.01,
    0.01,
    0.01,
]) ** 2)

In [ ]:
# Set up history arrays for plotting.
true_history = np.zeros((number_of_steps, 4))
measurement_history = np.zeros((number_of_steps, 1))
estimate_history = np.zeros((number_of_steps, 4))
covariance_history = np.zeros((number_of_steps, 4, 4))

(
    fig,
    axis,
    display_handle,
    measurement_scatter,
    true_line,
    estimate_line,
    estimate_point,
    heading_line,
    pose_artists,
) = configure_live_plot(
    reference_x,
    reference_y,
    "EKF with velocity-only measurements\nEllipses show 2-sigma XY covariance",
    show_measurements=False,
)

update_every = 3

# Start driving the robot and running the EKF.
for k in range(number_of_steps):
    control = np.array([
        commanded_speed[k],
        commanded_heading[k],
    ])

    # Simulated true robot dynamics.
    true_state = motion_model(
        true_state,
        control,
        dt,
        true_alpha_speed,
        true_alpha_heading,
    )
    true_state += rng.normal(0.0, true_process_std)
    true_state[2] = wrap_angle(true_state[2])

    # The EKF receives only a noisy speed measurement.
    measurement = velocity_measurement_model(true_state) + rng.normal(
        0.0, velocity_measurement_std
    )

    # EKF prediction step
    estimated_mean, estimated_covariance = ekf_predict(
        estimated_mean,
        estimated_covariance,
        control,
        dt,
        model_alpha_speed,
        model_alpha_heading,
        process_noise,
    )

    # EKF update step
    estimated_mean, estimated_covariance = ekf_update(
        estimated_mean,
        estimated_covariance,
        measurement,
        velocity_measurement_noise,
        velocity_measurement_model,
        velocity_measurement_jacobian,
    )

    # Plotting and history storage.
    true_history[k] = true_state
    measurement_history[k] = measurement
    estimate_history[k] = estimated_mean
    covariance_history[k] = estimated_covariance

    if k % update_every == 0 or k == number_of_steps - 1:
        true_line.set_data(
            true_history[:k + 1, 0], true_history[:k + 1, 1]
        )
        estimate_line.set_data(
            estimate_history[:k + 1, 0], estimate_history[:k + 1, 1]
        )
        estimate_point.set_data(
            [estimated_mean[0]], [estimated_mean[1]]
        )

        update_pose_artists(
            axis,
            heading_line,
            pose_artists,
            estimated_mean,
            estimated_covariance,
        )
        display_handle.update(fig)


#### Exercise: 

Calibrate the uncertainty to make it as low as possible, while still capturing the true trajectory
